Training XOR function with CHL in single compartment

# JAX package installation

In [ ]:
!pip install "jax==0.7.2" "jaxlib==0.7.2" "diffrax==0.7.0" "equinox" "optax"

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, lax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from tqdm.auto import tqdm
import optax
import time

jax.config.update("jax_enable_x64", True)

# ---------- Colab-friendly import of shared Dynamics/Training modules ----------
import os, sys

REPO_URL = "https://github.com/YanchengDu/LiquidCHL.git"
REPO_DIR = "LiquidCHL"

if os.path.isdir("Dynamics") and os.path.isdir("Training"):
    repo_root = "."
elif os.path.isdir(os.path.join("..", "Dynamics")):
    repo_root = ".."
else:
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    repo_root = REPO_DIR

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from Dynamics.Model_A import forward_sim_x_ssolvent_clamp, x_to_phi
from Training.contrastive_hebbian_learning_classifier import (
    daydreaming_contrastive_hebbian_learning_lagrange_JAX_fast_sample_adamw,
    CHL_training_hidden,
    evaluate_and_plot_training,
    _plot_grayscale_blocks,
)

In [ ]:
# ---------- Shared config: dynamics (training + testing) and training hyperparameters ----------
# Use these in one place so training and testing stay consistent.
from random import randint


DT_DYNAMICS = 1e-1
MAX_STEPS = 30_000
T_END = 300.0  # Integration end time (used by training and testing)

# Training hyperparameters (used by CHL_training_hidden / daydreaming)
GAMMA_LEARNING = 0.02
N_SAMPLE = 500
P_BOUNDARY = 0.3
WIDTH = 0.05
# Seed for training (set to an int for reproducibility, or None for random)
TRAINING_SEED = randint(0, 1000)
# Model / run layout (used by training cells and parameter sweep)
N_SPECIES = 11
CLAMPED = 4

# Contrastive hebbian learning

In [ ]:
# ---------------- XOR-specific random-memory generator ----------------
# XOR rule: output is "high" iff the two inputs are on the SAME side of 0.125
# (both above, or both below) -- i.e. same-sign -> high Output1, different-sign -> high Output2.
def make_random_memories(key, sample_size, N):
    keys = jax.random.split(key, sample_size + 1)
    key_next = keys[0]
    mem_keys = keys[1:]

    def make_one(k):
        k1, k2, k3, k4 = jax.random.split(k, 4)
        m1 = jax.random.uniform(k3, ()) < P_BOUNDARY
        m2 = jax.random.uniform(k4, ()) < 0.5

        # sample uniform
        inp0_u = jax.random.uniform(k1, shape=(), minval=0.0, maxval=0.25)
        inp1_u = jax.random.uniform(k2, shape=(), minval=0.0, maxval=0.25)
        # sample near boundary
        inp0_b1 = jax.random.uniform(k1, shape=(), minval=0.125 - WIDTH, maxval=0.125 + WIDTH)
        inp1_b1 = jax.random.uniform(k2, shape=(), minval=0.0, maxval=0.25)

        inp0_b = jnp.where(m2, inp0_b1, inp1_b1)
        inp1_b = jnp.where(m2, inp1_b1, inp0_b1)

        inp0 = jnp.where(m1, inp0_u, inp0_b)
        inp1 = jnp.where(m1, inp1_u, inp1_b)

        out = jnp.zeros(N)
        out = out.at[0].set(inp0).at[1].set(inp1)

        cond = ((inp0 > 0.125) & (inp1 > 0.125)) | ((inp0 < 0.125) & (inp1 < 0.125))
        hi, lo = 1.1 / N, 0.25 / N
        val2 = jnp.where(cond, hi, lo)
        val3 = jnp.where(cond, lo, hi)
        out = out.at[2].set(val2).at[3].set(val3)

        sum_fixed = jnp.sum(out[:4])
        sum_rem = jnp.sum(out[4:])
        out = out.at[4:].set(out[4:] / sum_rem * (1 - sum_fixed))
        return out

    mems = jax.vmap(make_one)(mem_keys)
    return key_next, mems


# XOR ground-truth boundary rule, used for accuracy reporting in evaluate_and_plot_training
def is_above_boundary(x, y):
    return ((x > 0.125) & (y > 0.125)) | ((x < 0.125) & (y < 0.125))

# Training evaluation

In [ ]:
def sample_training_memories(key_seed, sample_size, N, p_boundary, width):
    """Generate one batch of training memories (same logic as daydreaming). Returns (n_sample, N) array."""
    key = jax.random.PRNGKey(key_seed)
    keys = jax.random.split(key, sample_size + 1)
    mem_keys = keys[1:]

    def make_one(k):
        k1, k2, k3, k4 = jax.random.split(k, 4)
        m1 = jax.random.uniform(k3, ()) < p_boundary
        m2 = jax.random.uniform(k4, ()) < 0.5

        inp0_u = jax.random.uniform(k1, shape=(), minval=0.0, maxval=0.25)
        inp1_u = jax.random.uniform(k2, shape=(), minval=0.0, maxval=0.25)
        inp0_b1 = jax.random.uniform(k1, shape=(), minval=0.125 - width, maxval=0.125 + width)
        inp1_b1 = jax.random.uniform(k2, shape=(), minval=0.0, maxval=0.25)

        inp0_b = jnp.where(m2, inp0_b1, inp1_b1)
        inp1_b = jnp.where(m2, inp1_b1, inp0_b1)

        inp0 = jnp.where(m1, inp0_u, inp0_b)
        inp1 = jnp.where(m1, inp1_u, inp1_b)

        out = jnp.zeros(N)
        out = out.at[0].set(inp0).at[1].set(inp1)
        sum_fixed = jnp.sum(out[:4])
        sum_rem = jnp.sum(out[4:])
        out = out.at[4:].set(out[4:] / sum_rem * (1 - sum_fixed))
        return out

    mems = jax.vmap(make_one)(mem_keys)
    return np.asarray(mems)


def plot_sampled_memories(mems, title='Sampled training memories (input space)'):
    """Scatter: (mems[:,0], mems[:,1]) colored by the XOR same-side/different-side rule."""
    mems = np.asarray(mems)
    x, y = mems[:, 0], mems[:, 1]
    above = (((x > 0.125) & (y > 0.125)) | ((x < 0.125) & (y < 0.125))).astype(int)
    plt.figure(figsize=(7, 6))
    plt.scatter(x, y, c=above, cmap='coolwarm', s=15, alpha=0.8)
    plt.colorbar(label='Same side (1) / Different side (0)')
    plt.xlabel('phi₀')
    plt.ylabel('phi₁')
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
mems = sample_training_memories(key_seed=TRAINING_SEED, sample_size=N_SAMPLE, N=N_SPECIES, p_boundary=P_BOUNDARY, width=WIDTH)
plot_sampled_memories(mems, title='My sampled memories')

# Training

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=1000, clamped=CLAMPED,
    make_random_memories_fn=make_random_memories,
    chi_prev=None, miu_prev=None,
    gamma_learning=GAMMA_LEARNING, n_sample=N_SAMPLE,
    seed=TRAINING_SEED,
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
    is_above_boundary=is_above_boundary,
)

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=1000, clamped=CLAMPED,
    make_random_memories_fn=make_random_memories,
    chi_prev=chi_learned, miu_prev=miu_learned,
    gamma_learning=GAMMA_LEARNING, n_sample=N_SAMPLE,
    seed=TRAINING_SEED,
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
    is_above_boundary=is_above_boundary,
)

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=1000, clamped=CLAMPED,
    make_random_memories_fn=make_random_memories,
    chi_prev=chi_learned, miu_prev=miu_learned,
    gamma_learning=GAMMA_LEARNING, n_sample=N_SAMPLE,
    seed=TRAINING_SEED,
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
    is_above_boundary=is_above_boundary,
)

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=3000, clamped=CLAMPED,
    make_random_memories_fn=make_random_memories,
    chi_prev=chi_learned, miu_prev=miu_learned,
    gamma_learning=GAMMA_LEARNING, n_sample=N_SAMPLE,
    seed=TRAINING_SEED,
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
    is_above_boundary=is_above_boundary,
)

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=3000, clamped=CLAMPED,
    make_random_memories_fn=make_random_memories,
    chi_prev=chi_learned, miu_prev=miu_learned,
    gamma_learning=GAMMA_LEARNING, n_sample=N_SAMPLE,
    seed=TRAINING_SEED,
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
    is_above_boundary=is_above_boundary,
)

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=3000, clamped=CLAMPED,
    make_random_memories_fn=make_random_memories,
    chi_prev=chi_learned, miu_prev=miu_learned,
    gamma_learning=GAMMA_LEARNING, n_sample=N_SAMPLE,
    seed=TRAINING_SEED,
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
    is_above_boundary=is_above_boundary,
)

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=3000, clamped=CLAMPED,
    make_random_memories_fn=make_random_memories,
    chi_prev=chi_learned, miu_prev=miu_learned,
    gamma_learning=GAMMA_LEARNING, n_sample=N_SAMPLE,
    seed=TRAINING_SEED,
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
    is_above_boundary=is_above_boundary,
)